# UA AST: Fine-tune Whisper + Baseline Comparison + COMET

This notebook runs the full experiment required for the assignment:

1. Load the Ukrainian parallel AST manifests.
2. Fine-tune Whisper on English audio -> Ukrainian text.
3. Generate Ukrainian hypotheses with the original baseline model.
4. Generate Ukrainian hypotheses with the fine-tuned model.
5. Write `src`, `ref`, and `hyp` files.
6. Compute COMET and compare baseline vs fine-tuned model.

The notebook is memory-safe for MacBook Air M4: batch size 1, lazy audio loading, frozen encoder, Adafactor, no generated validation during training, and explicit cleanup between large model loads.

## 0. Install Dependencies

Run this once if any imports fail. After installation, restart the kernel and run the notebook from the top.

In [ ]:
%pip install 'accelerate>=0.26.0' soundfile tqdm sacrebleu unbabel-comet

## 1. Imports

In [35]:
from pathlib import Path
import gc
import json
import os
from dataclasses import dataclass
from typing import Any

import numpy as np
import soundfile as sf
import torch
from torch.utils.data import Dataset as TorchDataset
from tqdm.auto import tqdm
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperFeatureExtractor,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperTokenizer,
)

## 2. Configuration

Use `RUN_MODE = 'smoke'` to verify that the notebook works. Use `RUN_MODE = 'final_full'` for your final experiment.

In [36]:
DATA_DIR = Path('ua_ast_data/manifests/combined_uk')
OUTPUT_DIR = Path('models/whisper-base-ua-ast')
EVAL_DIR = Path('ua_ast_eval')

BASE_MODEL = 'openai/whisper-base'

TRAIN_MANIFEST = DATA_DIR / 'train.jsonl'
VAL_MANIFEST = DATA_DIR / 'validation.clean.jsonl'
TEST_MANIFESTS = {
    'test.clean': DATA_DIR / 'test.clean.jsonl',
    'test.other': DATA_DIR / 'test.other.jsonl',
}

# Options: 'smoke' or 'final_full'
RUN_MODE = 'final_full'

if RUN_MODE == 'final_full':
    MAX_TRAIN_SAMPLES = None
    MAX_VAL_SAMPLES = None
    MAX_TEST_SAMPLES = None
    NUM_EPOCHS = 3
else:
    MAX_TRAIN_SAMPLES = 500
    MAX_VAL_SAMPLES = 50
    MAX_TEST_SAMPLES = 100
    NUM_EPOCHS = 1

BATCH_SIZE = 1
GRAD_ACCUM = 32
LEARNING_RATE = 3e-6
MAX_TARGET_LENGTH = 225
FREEZE_ENCODER = True
USE_CPU_ONLY = False  # set True only if MPS gives memory errors; CPU is much slower

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() and not USE_CPU_ONLY else 'cpu')
print('Run mode:', RUN_MODE)
print('Device:', DEVICE)
print('Train samples:', MAX_TRAIN_SAMPLES if MAX_TRAIN_SAMPLES is not None else 'ALL')
print('Val samples:', MAX_VAL_SAMPLES if MAX_VAL_SAMPLES is not None else 'ALL')
print('Test samples:', MAX_TEST_SAMPLES if MAX_TEST_SAMPLES is not None else 'ALL')
print('Epochs:', NUM_EPOCHS)

Run mode: final_full
Device: mps
Train samples: ALL
Val samples: ALL
Test samples: ALL
Epochs: 3


## 3. Helpers and Dataset

In [37]:
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        torch.mps.empty_cache()


def read_manifest(path: Path, max_samples: int | None = None, require_target: bool = True) -> list[dict]:
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            audio_path = record.get('audio_path', '')
            source_text = record.get('source_text', record.get('text', '')).strip()
            target_text = record.get('target_text_uk', record.get('target_text', '')).strip()
            if not Path(audio_path).exists():
                continue
            if require_target and not target_text:
                continue
            records.append({
                'id': record.get('id', str(len(records))),
                'audio_path': audio_path,
                'source_text': source_text,
                'target_text': target_text,
            })
            if max_samples is not None and len(records) >= max_samples:
                break
    return records


class ManifestSpeechDataset(TorchDataset):
    def __init__(self, records: list[dict]):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        audio, sr = sf.read(record['audio_path'], dtype='float32')
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        return {
            'audio': audio,
            'sampling_rate': sr,
            'target_text': record['target_text'],
        }


train_records = read_manifest(TRAIN_MANIFEST, MAX_TRAIN_SAMPLES)
val_records = read_manifest(VAL_MANIFEST, MAX_VAL_SAMPLES)
train_dataset = ManifestSpeechDataset(train_records)
val_dataset = ManifestSpeechDataset(val_records)

print('train:', len(train_dataset))
print('validation.clean:', len(val_dataset))

train: 28539
validation.clean: 2703


## 4. Load Processor and Model

In [38]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL)
tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL, language='Ukrainian', task='transcribe')
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language='Ukrainian', task='transcribe')

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language='Ukrainian', task='transcribe')
model.config.suppress_tokens = []
model.config.use_cache = False
model.generation_config.language = 'Ukrainian'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = model.config.forced_decoder_ids

model.gradient_checkpointing_enable()
if FREEZE_ENCODER:
    model.freeze_encoder()
    print('Encoder frozen to save memory')

cleanup_memory()

Encoder frozen to save memory


## 5. Data Collator

Important: Whisper requires mel features padded to length `3000`. This collator uses fixed padding to avoid the previous `found 1631` error.

In [39]:
@dataclass
class MemorySafeWhisperCollator:
    processor: Any

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        arrays = [feature['audio'] for feature in features]
        sampling_rates = {int(feature['sampling_rate']) for feature in features}
        if sampling_rates != {16000}:
            raise ValueError(f'Expected 16 kHz audio, got: {sampling_rates}')

        batch = self.processor.feature_extractor(
            arrays,
            sampling_rate=16000,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
        )

        labels = self.processor.tokenizer(
            [feature['target_text'] for feature in features],
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
        )
        label_ids = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        if (label_ids[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            label_ids = label_ids[:, 1:]
        batch['labels'] = label_ids
        return batch


collator = MemorySafeWhisperCollator(processor=processor)

# Fast sanity check before long training.
example_batch = collator([train_dataset[0]])
print('input_features:', example_batch['input_features'].shape)
print('labels:', example_batch['labels'].shape)

input_features: torch.Size([1, 80, 3000])
labels: torch.Size([1, 73])


## 6. Training Arguments

In [40]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    optim='adafactor',
    warmup_steps=25,
    num_train_epochs=NUM_EPOCHS,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    bf16=False,

    # Do not generate during training on Mac; COMET eval is done after saving.
    predict_with_generate=False,
    eval_strategy='no',

    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    load_best_model_at_end=False,
    use_cpu=USE_CPU_ONLY,
    torch_empty_cache_steps=25,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    processing_class=processor,
)

steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))
print('Approx optimizer steps per epoch:', steps_per_epoch)
print('Approx total optimizer steps:', steps_per_epoch * NUM_EPOCHS)

Approx optimizer steps per epoch: 891
Approx total optimizer steps: 2673


## 7. Train and Save

In [ ]:
cleanup_memory()
train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
cleanup_memory()
print('Saved fine-tuned model to', OUTPUT_DIR)
print(train_result)

## 8. Generation Helpers for Baseline and Fine-tuned Evaluation

In [ ]:
def load_audio(path: str) -> tuple[np.ndarray, int]:
    audio, sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    return audio, sr


def generate_hypotheses(model_name_or_path: str | Path, records: list[dict], desc: str) -> list[str]:
    local_processor = WhisperProcessor.from_pretrained(model_name_or_path, language='Ukrainian', task='transcribe')
    local_model = WhisperForConditionalGeneration.from_pretrained(model_name_or_path)
    local_model.config.forced_decoder_ids = local_processor.get_decoder_prompt_ids(language='Ukrainian', task='transcribe')
    local_model.config.suppress_tokens = []
    local_model.generation_config.language = 'Ukrainian'
    local_model.generation_config.task = 'transcribe'
    local_model.generation_config.forced_decoder_ids = local_model.config.forced_decoder_ids
    local_model.to(DEVICE)
    local_model.eval()

    outputs = []
    for record in tqdm(records, desc=desc):
        audio, sr = load_audio(record['audio_path'])
        if sr != 16000:
            raise ValueError(f"Expected 16 kHz audio, got {sr} for {record['audio_path']}")

        inputs = local_processor.feature_extractor(
            audio,
            sampling_rate=16000,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
        )
        input_features = inputs.input_features.to(DEVICE)

        with torch.no_grad():
            predicted_ids = local_model.generate(
                input_features,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=1,
                do_sample=False,
            )
        text = local_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
        outputs.append(text)

        del input_features, predicted_ids, inputs
        cleanup_memory()

    local_model.to('cpu')
    del local_model, local_processor
    cleanup_memory()
    return outputs


def write_lines(path: Path, lines: list[str]):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n'.join(line.replace('\n', ' ').strip() for line in lines) + '\n', encoding='utf-8')

## 9. Generate Baseline and Fine-tuned Outputs

This produces the required files for COMET:

- `src.txt`: original English text
- `ref.txt`: reference Ukrainian translation
- `hyp_baseline.txt`: baseline Whisper output
- `hyp_finetuned.txt`: fine-tuned Whisper output

In [ ]:
all_eval_results = {}

for split_name, manifest_path in TEST_MANIFESTS.items():
    print('\n===', split_name, '===')
    records = read_manifest(manifest_path, MAX_TEST_SAMPLES)
    split_dir = EVAL_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    src_lines = [r['source_text'] for r in records]
    ref_lines = [r['target_text'] for r in records]
    write_lines(split_dir / 'src.txt', src_lines)
    write_lines(split_dir / 'ref.txt', ref_lines)

    baseline_hyp = generate_hypotheses(BASE_MODEL, records, desc=f'baseline {split_name}')
    write_lines(split_dir / 'hyp_baseline.txt', baseline_hyp)

    finetuned_hyp = generate_hypotheses(OUTPUT_DIR, records, desc=f'fine-tuned {split_name}')
    write_lines(split_dir / 'hyp_finetuned.txt', finetuned_hyp)

    all_eval_results[split_name] = {
        'records': len(records),
        'src': split_dir / 'src.txt',
        'ref': split_dir / 'ref.txt',
        'hyp_baseline': split_dir / 'hyp_baseline.txt',
        'hyp_finetuned': split_dir / 'hyp_finetuned.txt',
    }
    print('Wrote files to', split_dir)

all_eval_results

## 10. COMET Evaluation

COMET can be slow and may download the model on first run. If RAM becomes tight, restart the kernel and run only the import/config/helper/evaluation cells.

In [41]:
# COMET may try to call Hugging Face even when files are cached.
# Offline mode avoids network errors after the model/tokenizer are already downloaded.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

from comet import download_model, load_from_checkpoint

comet_model_path = download_model('Unbabel/wmt22-comet-da')
comet_model = load_from_checkpoint(comet_model_path)


def read_lines(path: Path) -> list[str]:
    return path.read_text(encoding='utf-8').splitlines()


def score_comet(src_path: Path, ref_path: Path, hyp_path: Path, desc: str) -> float:
    src = read_lines(src_path)
    ref = read_lines(ref_path)
    hyp = read_lines(hyp_path)
    data = [
        {'src': s, 'mt': h, 'ref': r}
        for s, h, r in zip(src, hyp, ref)
    ]
    result = comet_model.predict(
        data,
        batch_size=1,
        gpus=0,
        progress_bar=True,
        num_workers=2 if torch.backends.mps.is_available() else 0,
    )
    score = float(result.system_score)
    print(f'{desc}: {score:.4f}')
    return score


report = {
    'run_mode': RUN_MODE,
    'base_model': BASE_MODEL,
    'finetuned_model': str(OUTPUT_DIR),
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'max_test_samples': MAX_TEST_SAMPLES,
    'num_epochs': NUM_EPOCHS,
    'splits': {},
}

for split_name, paths in all_eval_results.items():
    baseline_score = score_comet(paths['src'], paths['ref'], paths['hyp_baseline'], f'{split_name} baseline')
    finetuned_score = score_comet(paths['src'], paths['ref'], paths['hyp_finetuned'], f'{split_name} fine-tuned')
    report['splits'][split_name] = {
        'records': paths['records'],
        'baseline_comet': baseline_score,
        'finetuned_comet': finetuned_score,
        'delta_comet': finetuned_score - baseline_score,
    }

report_path = EVAL_DIR / 'comet_report_full.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
report

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 53227.21it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try inst

test.clean baseline: 0.3209


/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 2620/2620 [07:00<00:00,  6.24it/s]
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs a

test.clean fine-tuned: 0.4174


/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 2939/2939 [08:23<00:00,  5.84it/s]
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs a

test.other baseline: 0.3266


/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 2939/2939 [08:07<00:00,  6.03it/s]

test.other fine-tuned: 0.4139


{'run_mode': 'final_full',
 'base_model': 'openai/whisper-base',
 'finetuned_model': 'models/whisper-base-ua-ast',
 'max_train_samples': None,
 'max_test_samples': None,
 'num_epochs': 3,
 'splits': {'test.clean': {'records': 2620,
   'baseline_comet': 0.3209238578714715,
   'finetuned_comet': 0.41744581426827965,
   'delta_comet': 0.09652195639680816},
  'test.other': {'records': 2939,
   'baseline_comet': 0.3265953166115625,
   'finetuned_comet': 0.4138608982811253,
   'delta_comet': 0.08726558166956283}}}

## 11. Final Summary Table

In [42]:
print('Split       | Records | Baseline COMET | Fine-tuned COMET | Delta')
print('------------|---------|----------------|------------------|-------')
for split_name, metrics in report['splits'].items():
    print(
        f"{split_name:<11} | "
        f"{metrics['records']:>7} | "
        f"{metrics['baseline_comet']:.4f}         | "
        f"{metrics['finetuned_comet']:.4f}           | "
        f"{metrics['delta_comet']:+.4f}"
    )

Split       | Records | Baseline COMET | Fine-tuned COMET | Delta
------------|---------|----------------|------------------|-------
test.clean  |    2620 | 0.3209         | 0.4174           | +0.0965
test.other  |    2939 | 0.3266         | 0.4139           | +0.0873


In [45]:
import random

def print_evaluation_examples(num_examples=5, split_name='toronto'):
    print(f"--- Examples for {split_name} ---")
    split_dir = EVAL_DIR / split_name
    
    if not (split_dir / 'hyp_finetuned.txt').exists():
        print(f"Fine-tuned hypothesis file not found for {split_name}. Run the evaluation cells first.")
        return
        
    src_lines = (split_dir / 'src.txt').read_text(encoding='utf-8').splitlines()
    ref_lines = (split_dir / 'ref.txt').read_text(encoding='utf-8').splitlines()
    hyp_base = (split_dir / 'hyp_baseline.txt').read_text(encoding='utf-8').splitlines()
    hyp_fine = (split_dir / 'hyp_finetuned.txt').read_text(encoding='utf-8').splitlines()
    
    num_total = len(src_lines)
    if num_total == 0:
        print("No evaluation records found.")
        return
        
    indices = random.sample(range(num_total), min(num_examples, num_total))
    
    for idx in indices:
        print(f"English Source (src): {src_lines[idx]}")
        print(f"Target Ukrainian (ref): {ref_lines[idx]}")
        print(f"Baseline Whisper (hyp): {hyp_base[idx] if idx < len(hyp_base) else 'N/A'}")
        print(f"Fine-tuned Whisper (hyp): {hyp_fine[idx] if idx < len(hyp_fine) else 'N/A'}")
        print("-" * 50)

# Print examples for each evaluated split
for split in TEST_MANIFESTS.keys():
    print_evaluation_examples(5, split)
    print("\n")

--- Examples for test.clean ---
English Source (src): my heart doth plead that thou in him dost lie a closet never pierc'd with crystal eyes but the defendant doth that plea deny and says in him thy fair appearance lies
Target Ukrainian (ref): Моє серце запевняє, що ти в ній лежить гардероб ніколи не пробито кристалевими очима, але обвинувачений заперечує, що запевняється і говорить, що у ній твоя гарна зовнішність брехне.
Baseline Whisper (hyp): Майхарт дозплід, that thou in him dоз lie. A closet never pierced with crystal eyes. But the defendant doth that plea deny, and says in him thy fair appearance lies.
Fine-tuned Whisper (hyp): Мій жахливий дозвол, що вона вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не вона не 